# Implement Multi-Query Attention from Scratch
### Problem Statement
Multi-Query Attention (MQA), introduced in [Shazeer 2019](https://arxiv.org/abs/1911.02150), is a memory-efficient variant of Multi-Head Attention. Instead of having separate K and V projections per head, MQA uses a **single shared K/V head** across all query heads.

This dramatically reduces the KV cache size during autoregressive inference (from `num_heads × d_head` to just `1 × d_head` per token), making it much faster for long-sequence generation.

Your goal is to implement MQA from scratch using PyTorch and validate it against standard Multi-Head Attention in the degenerate case where `num_heads = 1`.

---

### Requirements

1. **Linear Projections**
   - Q projection: `d_model → d_model` (all query heads)
   - K projection: `d_model → d_head` (single shared head)
   - V projection: `d_model → d_head` (single shared head)
   - Output projection: `d_model → d_model`

2. **Expand K/V to Match Query Heads**
   - Use `.expand()` to broadcast the single K/V head across all query heads.

3. **Scaled Dot-Product Attention**
   - Compute: `scores = Q @ Kᵀ / sqrt(d_head)`
   - Apply optional mask before softmax.
   - Weight V with attention weights.

4. **Combine Heads**
   - Concatenate all query head outputs.
   - Apply final linear projection.

5. **Validate Against MHA**
   - When `num_heads = 1`, MQA is identical to single-head MHA.
   - Copy weights and verify outputs match with `torch.allclose()`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ K and V must be projected to a single head (`d_head`), not `d_model`.
- ✅ Support optional masking.
- ✅ Must match `torch.nn.MultiheadAttention` output when `num_heads = 1`.

---

<details>
  <summary>💡 Hint</summary>

  - Q shape after projection and reshape: `(batch, num_heads, seq_len, d_head)`
  - K/V shape after projection and reshape: `(batch, 1, seq_len, d_head)`
  - Use `.expand(-1, num_heads, -1, -1)` to broadcast K/V to all heads without copying memory.
  - When `num_heads = 1`, the Q projection is `d_model → d_model` which equals `d_model → 1 * d_head = d_head = d_model`, so all projections have the same shape — identical to single-head MHA.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

In [ ]:
class MultiQueryAttention(nn.Module):
    """
    Implements Multi-Query Attention (MQA).
    All query heads share a single key/value head.
    """
    def __init__(self, num_heads, d_model):
        super().__init__()
        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.d_model = d_model
        self.d_head = d_model // num_heads

        # Q projects to all heads, K/V project to a single shared head
        self.Q_w = nn.Linear(d_model, d_model, bias=False)            # d_model -> num_heads * d_head
        self.K_w = nn.Linear(d_model, self.d_head, bias=False)        # d_model -> 1 * d_head
        self.V_w = nn.Linear(d_model, self.d_head, bias=False)        # d_model -> 1 * d_head
        self.W_out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, q, k, v, mask=None):
        """
        Args:
            q (Tensor): Query tensor of shape (batch_size, seq_len, d_model)
            k (Tensor): Key tensor of shape (batch_size, seq_len, d_model)
            v (Tensor): Value tensor of shape (batch_size, seq_len, d_model)
            mask (Tensor, optional): Masking tensor for attention

        Returns:
            Tensor: MQA output of shape (batch_size, seq_len, d_model)
        """
        batch_size, seq_len, _ = q.shape

        # Project Q to all heads, K/V to single head
        Q = self.Q_w(q)  # (batch_size, seq_len, d_model)
        K = self.K_w(k)  # (batch_size, seq_len, d_head)
        V = self.V_w(v)  # (batch_size, seq_len, d_head)

        # Reshape Q to (batch, num_heads, seq_len, d_head)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        # Reshape K, V to (batch, 1, seq_len, d_head)
        K = K.view(batch_size, seq_len, 1, self.d_head).transpose(1, 2)
        V = V.view(batch_size, seq_len, 1, self.d_head).transpose(1, 2)

        # Expand K, V to match num_heads (no memory copy — just broadcasting)
        K = K.expand(-1, self.num_heads, -1, -1)  # (batch, num_heads, seq_len, d_head)
        V = V.expand(-1, self.num_heads, -1, -1)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_head ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)  # (batch, num_heads, seq_len, d_head)

        # Combine heads
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        return self.W_out(output)

## 📚 MQA vs MHA: What Changed?

| | MHA | MQA |
|---|---|---|
| Q projection | `d_model → d_model` | `d_model → d_model` (same) |
| K projection | `d_model → d_model` | `d_model → d_head` |
| V projection | `d_model → d_model` | `d_model → d_head` |
| K/V shape | `(B, num_heads, S, d_head)` | `(B, 1, S, d_head)` → expanded |
| KV cache per token | `num_heads × d_head × 2` | `d_head × 2` |
| Parameter count | `3 × d_model²` + output | `d_model² + 2 × d_model × d_head` + output |

### Why does this work?

The key insight is that **different query heads can still learn different attention patterns** even though they attend over the same K/V representations. The Q projection provides all the head-specific specialization, while K/V provide a shared "memory" that all heads read from.

### The expand trick

```python
K = K.expand(-1, self.num_heads, -1, -1)
```

`.expand()` doesn't copy data — it creates a view where the same memory is referenced by all heads. This is both memory-efficient and semantically correct: all heads truly share the same K/V.

Compare with `.repeat()` which actually copies the data:
```python
# expand: no copy, just a strided view (preferred)
K.expand(-1, num_heads, -1, -1)  
# repeat: copies data num_heads times (wastes memory)
K.repeat(1, num_heads, 1, 1)
```

In [ ]:
# Validate: when num_heads=1, MQA == single-head MHA
# (because there's only 1 query head and 1 KV head — identical)
num_heads_test = 1

multihead_attn = torch.nn.MultiheadAttention(
    embed_dim=d_model, num_heads=num_heads_test, bias=False, batch_first=True
)

custom_attn = MultiQueryAttention(num_heads_test, d_model)

# Copy weights: when num_heads=1, Q/K/V projections are all d_model -> d_model
custom_attn.Q_w.weight.data = multihead_attn.in_proj_weight[:d_model, :].clone()
custom_attn.K_w.weight.data = multihead_attn.in_proj_weight[d_model:2*d_model, :].clone()
custom_attn.V_w.weight.data = multihead_attn.in_proj_weight[2*d_model:, :].clone()
custom_attn.W_out.weight.data = multihead_attn.out_proj.weight.data.clone()

output_custom = custom_attn(q, k, v)
output_ref, _ = multihead_attn(q, k, v)

print("Custom MQA output:")
print(output_custom)
print("\nPyTorch MHA output (num_heads=1):")
print(output_ref)

assert torch.allclose(output_custom, output_ref, atol=1e-06, rtol=1e-05)
print("\n✅ Outputs match! MQA implementation is correct.")

In [ ]:
# Bonus: verify KV cache savings
# MHA KV cache per token: num_heads * d_head * 2 (K and V)
# MQA KV cache per token: 1 * d_head * 2 (K and V)
print(f"MHA KV cache per token: {num_heads * (d_model // num_heads) * 2} values")
print(f"MQA KV cache per token: {1 * (d_model // num_heads) * 2} values")
print(f"Memory reduction: {num_heads}x")